In [1]:
import glob, json, re
from urllib.parse import urlparse
import numpy as np
import pandas as pd

data = pd.concat(
    [pd.read_csv(f) for f in sorted(glob.glob("output/articles_clean_part*.csv.zip"))],
    ignore_index=True,
)
print("Shape:", data.shape)

CONTRACT = {
    "required_columns": ["url", "title", "content", "source"],
    "optional_columns": ["date_of_publication", "author_name", "description"],
    "min_title_chars_warning": 8,
    "min_content_chars_warning": 200,
    "known_error_patterns": [r"access denied", r"enable javascript", r"captcha",
                             r"robot check", r"403 forbidden", r"page not found"],
}
print(json.dumps(CONTRACT, indent=2))

Shape: (97241, 7)
{
  "required_columns": [
    "url",
    "title",
    "content",
    "source"
  ],
  "optional_columns": [
    "date_of_publication",
    "author_name",
    "description"
  ],
  "min_title_chars_warning": 8,
  "min_content_chars_warning": 200,
  "known_error_patterns": [
    "access denied",
    "enable javascript",
    "captcha",
    "robot check",
    "403 forbidden",
    "page not found"
  ]
}


In [2]:
df = data.copy().reset_index(drop=True)
df["row_index"] = df.index
issues = []

def add_issue(rule_id, severity, row_index, field, message):
    issues.append({"rule_id": rule_id, "severity": severity, "row_index": int(row_index),
                   "field": field, "message": message})

def is_valid_http_url(v):
    if not isinstance(v, str) or not v.strip():
        return False
    try:
        p = urlparse(v.strip())
        return p.scheme.lower() in {"http", "https"} and bool(p.hostname) and "." in p.hostname
    except Exception:
        return False

# schema
for col in CONTRACT["required_columns"]:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# required fields (vectorized)
for field in CONTRACT["required_columns"]:
    bad = df[field].isna() | (df[field].astype(str).str.strip() == "")
    for i in df.index[bad]:
        add_issue("FIELD_001", "ERROR", i, field, f"Required field '{field}' is missing or blank.")

# URL
bad_url = ~df["url"].map(is_valid_http_url)
for i in df.index[bad_url]:
    add_issue("URL_001", "ERROR", i, "url", "URL is not a plausible HTTP(S) URL.")

# dates
pub = pd.to_datetime(df["date_of_publication"], errors="coerce", utc=True, format="mixed")
for i in df.index[df["date_of_publication"].notna() & pub.isna()]:
    add_issue("TIME_002", "WARNING", i, "date_of_publication", "Timestamp cannot be parsed.")
for i in df.index[pub > pd.Timestamp("2026-10-29", tz="UTC")]:
    add_issue("TIME_003", "WARNING", i, "date_of_publication", "Publication date is in the future.")

# text
tl = df["title"].fillna("").astype(str).str.len()
cl = df["content"].fillna("").astype(str).str.len()
for i in df.index[(tl > 0) & (tl < CONTRACT["min_title_chars_warning"])]:
    add_issue("TEXT_001", "WARNING", i, "title", "Title is very short.")
for i in df.index[(cl > 0) & (cl < CONTRACT["min_content_chars_warning"])]:
    add_issue("TEXT_002", "WARNING", i, "content", "Content is very short.")
err = re.compile("|".join(CONTRACT["known_error_patterns"]), re.I)
blocked = df["content"].fillna("").astype(str).str.contains(err)
for i in df.index[blocked]:
    add_issue("TEXT_003", "WARNING", i, "content", "Content may be a block/CAPTCHA/error page.")

# duplicates
for i in df.index[df["url"].duplicated(keep=False)]:
    add_issue("DUP_001", "WARNING", i, "url", "URL occurs more than once.")
for i in df.index[df["title"].duplicated(keep=False)]:
    add_issue("DUP_003", "WARNING", i, "title", "Title occurs more than once; inspect.")
for i in df.index[df["content"].duplicated(keep=False)]:
    add_issue("DUP_002", "WARNING", i, "content", "Content occurs more than once.")

issues_df = pd.DataFrame(issues, columns=["rule_id", "severity", "row_index", "field", "message"])
print("Total issues:", len(issues_df))
print(issues_df.groupby(["rule_id", "severity"]).size().to_string() if len(issues_df) else "No issues")

Total issues: 670
rule_id   severity
DUP_003   WARNING     617
TEXT_001  WARNING       1
TEXT_003  WARNING      50
TIME_003  WARNING       2


In [3]:
import os

FATAL_RULES = {"FIELD_001", "URL_001"}
row_rules = issues_df.groupby("row_index")["rule_id"].apply(list).to_dict()

def disposition(i):
    rules = row_rules.get(i, [])
    if any(r in FATAL_RULES for r in rules):
        return "QUARANTINE"
    return "WARNING" if rules else "ACCEPT"

df["validation_issue_codes"] = df["row_index"].map(lambda i: ";".join(row_rules.get(i, [])))
df["validation_disposition"] = df["row_index"].map(disposition)

accepted_df = df[df["validation_disposition"] == "ACCEPT"]
warning_df = df[df["validation_disposition"] == "WARNING"]
quarantine_df = df[df["validation_disposition"] == "QUARANTINE"]
print({"total": len(df), "accepted": len(accepted_df),
       "warning": len(warning_df), "quarantine": len(quarantine_df)})

# --- unit tests ---
def test_valid_url():
    assert is_valid_http_url("https://example.com/story/1")
    assert not is_valid_http_url("not-a-url")
    assert not is_valid_http_url("ftp://example.com/file")
    assert not is_valid_http_url(None)

def test_timestamp():
    assert pd.notna(pd.to_datetime("2026-10-27T09:30:00Z", errors="coerce", utc=True))
    assert pd.isna(pd.to_datetime("yesterday maybe", errors="coerce", utc=True))

def test_no_fatal_rows():
    assert len(quarantine_df) == 0

test_valid_url(); test_timestamp(); test_no_fatal_rows()
print("All tests passed.")

# --- summary + exports ---
summary = {
    "input_rows": int(len(df)),
    "total_issues": int(len(issues_df)),
    "issues_by_rule": {k: int(v) for k, v in issues_df.groupby("rule_id").size().items()},
    "dispositions": {"accept": len(accepted_df), "warning": len(warning_df),
                     "quarantine": len(quarantine_df)},
}
os.makedirs("output/validation", exist_ok=True)
issues_df.to_csv("output/validation/news_validation_issues.csv", index=False, encoding="utf-8-sig")
warning_df[["row_index", "source", "url", "title", "validation_issue_codes"]].to_csv(
    "output/validation/news_warning_review.csv", index=False, encoding="utf-8-sig")
quarantine_df[["row_index", "source", "url", "title", "validation_issue_codes"]].to_csv(
    "output/validation/news_quarantine.csv", index=False, encoding="utf-8-sig")
with open("output/validation/news_validation_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
for p in sorted(os.listdir("output/validation")):
    print(p, round(os.path.getsize(f"output/validation/{p}") / 1024**2, 2), "MB")

{'total': 97241, 'accepted': 96571, 'warning': 670, 'quarantine': 0}
All tests passed.
{
  "input_rows": 97241,
  "total_issues": 670,
  "issues_by_rule": {
    "DUP_003": 617,
    "TEXT_001": 1,
    "TEXT_003": 50,
    "TIME_003": 2
  },
  "dispositions": {
    "accept": 96571,
    "warning": 670,
    "quarantine": 0
  }
}
news_quarantine.csv 0.0 MB
news_validation_issues.csv 0.04 MB
news_validation_summary.json 0.0 MB
news_warning_review.csv 0.11 MB


In [1]:
import glob, re, json, math
import pandas as pd
from urllib.parse import urlparse

OUT = "output"
res = []
def check(name, ok, detail=""):
    res.append(ok)
    print(("PASS" if ok else "FAIL"), "|", name, "|", detail)

df = pd.read_csv(f"{OUT}/articles_clean.csv")
rej = pd.read_csv(f"{OUT}/news_rejected.csv")
parts = sorted(glob.glob(f"{OUT}/articles_clean_part*.csv.zip"))
parts_df = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
summ = json.load(open(f"{OUT}/news_preprocessing_summary.json"))

cols = ["title","description","date_of_publication","author_name","content","url","source"]
check("columns", list(df.columns) == cols, str(df.shape))
check("count 100000 = clean + rejected", len(df) + len(rej) == 100000, f"{len(df)}+{len(rej)}")
check("summary json matches", summ["clean_rows"] == len(df) and summ["rejected_rows"] == len(rej))
check("5 zip parts", len(parts) == 5 and len(parts_df) == len(df) and parts_df["url"].equals(df["url"]),
      f"{len(parts)} parts, {len(parts_df)} rows")

for c in ["title","content","url","source"]:
    check(f"no null {c}", df[c].isna().sum() == 0, f"{df[c].isna().sum()} null")
check("url unique", df["url"].duplicated().sum() == 0)
check("title unique", df["title"].duplicated().sum() == 0)
check("content unique", df["content"].duplicated().sum() == 0)
norm = df["title"].str.lower().str.replace(r"[^a-z0-9 ]", "", regex=True).str.strip()
check("title unique (normalized)", norm.duplicated().sum() == 0, f"{norm.duplicated().sum()} dup")

def ok_url(u):
    p = urlparse(str(u)); return p.scheme in ("http","https") and "." in (p.hostname or "")
check("urls valid", df["url"].map(ok_url).all())

for c in ["title","description","author_name","content"]:
    s = df[c].dropna().astype(str)
    check(f"{c}: no HTML tags", not s.str.contains(r"<[a-zA-Z/!][^>]*>").any())
    check(f"{c}: no invisible/double spaces", not s.str.contains("[\u200b\u200c\u200d\ufeff\u00ad]|\\s{2,}").any())
    check(f"{c}: no &amp; entities", not s.str.contains(r"&(amp|nbsp|quot|#\d+);").any())

wc = df["content"].str.split().str.len()
check("content >= 30 words", (wc >= 30).all(), f"min={wc.min()}, median={wc.median()}, max={wc.max()}")
check("content <= 5000 words", (wc <= 5000).all())
check("title length 3-300", df["title"].str.len().between(3, 300).all())

d = pd.to_datetime(df["date_of_publication"], errors="coerce", utc=True, format="mixed")
check("dates parse", d.isna().mean() < 0.01, f"{d.isna().sum()} bad")
check("no future dates", (d.dropna() <= pd.Timestamp.now(tz="UTC")).all(), f"{d.min()} -> {d.max()}")

check("no puzzle titles", not df["title"].str.contains("Wordle|Connections|Strands|Mini Crossword|Spelling Bee", case=False).any())
blk = df["content"].str.contains("access denied|enable javascript|captcha|403 forbidden|subscribe to continue", case=False) & (df["content"].str.len() < 500)
check("no block/paywall pages", not blk.any(), f"{blk.sum()} suspicious")
check("no 'unknown/n/a' authors", not df["author_name"].dropna().str.lower().isin(["unknown","n/a","none","null"]).any())

share = df["source"].value_counts(normalize=True)
print("\nauthor null %:", round(df["author_name"].isna().mean()*100, 1))
print("Top source share % (imbalance check):\n", (share.head(5)*100).round(1).to_string())
check("no source > 40%", share.iloc[0] < 0.40, f"top={share.index[0]} {share.iloc[0]:.1%}")

print(f"\nRESULT: {sum(res)}/{len(res)} passed")

PASS | columns | (96732, 7)
PASS | count 100000 = clean + rejected | 96732+3268
PASS | summary json matches | 
PASS | 5 zip parts | 5 parts, 96732 rows
PASS | no null title | 0 null
PASS | no null content | 0 null
PASS | no null url | 0 null
PASS | no null source | 0 null
PASS | url unique | 
PASS | title unique | 
PASS | content unique | 
FAIL | title unique (normalized) | 19 dup
PASS | urls valid | 
PASS | title: no HTML tags | 
PASS | title: no invisible/double spaces | 
PASS | title: no &amp; entities | 
PASS | description: no HTML tags | 


C:\Users\SamTech\AppData\Local\Temp\ipykernel_21528\3072235725.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  check(f"{c}: no &amp; entities", not s.str.contains(r"&(amp|nbsp|quot|#\d+);").any())


PASS | description: no invisible/double spaces | 
PASS | description: no &amp; entities | 
PASS | author_name: no HTML tags | 


C:\Users\SamTech\AppData\Local\Temp\ipykernel_21528\3072235725.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  check(f"{c}: no &amp; entities", not s.str.contains(r"&(amp|nbsp|quot|#\d+);").any())


PASS | author_name: no invisible/double spaces | 
PASS | author_name: no &amp; entities | 


C:\Users\SamTech\AppData\Local\Temp\ipykernel_21528\3072235725.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  check(f"{c}: no &amp; entities", not s.str.contains(r"&(amp|nbsp|quot|#\d+);").any())


PASS | content: no HTML tags | 
PASS | content: no invisible/double spaces | 


C:\Users\SamTech\AppData\Local\Temp\ipykernel_21528\3072235725.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  check(f"{c}: no &amp; entities", not s.str.contains(r"&(amp|nbsp|quot|#\d+);").any())


PASS | content: no &amp; entities | 
PASS | content >= 30 words | min=80, median=569.0, max=5071
FAIL | content <= 5000 words | 
PASS | title length 3-300 | 
PASS | dates parse | 0 bad
PASS | no future dates | 2019-11-15 00:00:00+00:00 -> 2026-10-30 00:00:00+00:00
PASS | no puzzle titles | 
PASS | no block/paywall pages | 0 suspicious
PASS | no 'unknown/n/a' authors | 

author null %: 2.6
Top source share % (imbalance check):
 source
foxnews        6.7
zdnet          6.7
moscowtimes    6.7
scmp           6.7
engadget       6.7
PASS | no source > 40% | top=foxnews 6.7%

RESULT: 32/34 passed
